# 03 · Ajuste fino (CNN y transformer) con validación por artículo

**Dónde correrlo:** una sesión de Jupyter en una cola con GPU (`nukwa-debug` para probar; `nukwa-l40s` para la corrida larga). **Kernel:** `Python-ML-DL` (PyTorch 2.3 con CUDA, timm 1.0.20). **Recursos sugeridos:** 4–8 núcleos y 16–32 GB.

**Antes de empezar:** el cuaderno 02 ya dejó las imágenes en `imagenes/` y `datos/tabla_modelo.csv`. Si no, ejecuta primero sus celdas 1 y 2.

**Qué hace:** ajusta fino dos modelos preentrenados (`resnet50` = CNN, `dinov2_vits14` = transformer) para predecir la clase de portador desde la figura, con las **5 particiones por artículo** (`fold`). Aumentos de datos en la GPU, pérdida con pesos por clase, precisión mixta, tasa de aprendizaje por capas y **punto de control en cada época** (si la cola corta o un administrador interrumpe, vuelve a ejecutar y retoma). No se elige nada con los datos de prueba: el número de épocas es fijo y solo se reporta la última época.

**Referencia a superar** (mismas particiones, vectores congelados): exactitud 0,485 / F1 macro 0,466 (imagen sola, SVM) y 0,454 de clase mayoritaria.

**Tiempo:** unos 2–4 minutos por partición y modelo en una L40S; en `nukwa-debug` (≈1 h) caben los dos modelos con las 5 particiones. El cuaderno se detiene solo antes del límite de tiempo (`TIEMPO_MAX_MIN`).

Salidas pequeñas en `resultados/`: `ft_<modelo>_fold<k>.csv` (predicciones y probabilidades), `metricas_ft_<modelo>.json`.

In [ ]:
# Celda 1: GPU, datos y carga de imágenes en memoria
import os, sys, json, time, math, random, socket, platform
from pathlib import Path
BASE = Path(os.environ.get("N2P_BASE", Path.home() / "ondemand" / "Kabre_Nano2Predict"))
DATOS, SALIDA, IMG = BASE / "datos", BASE / "resultados", BASE / "imagenes"
SALIDA.mkdir(exist_ok=True)
T_INICIO = time.time()

import numpy as np, pandas as pd, torch, timm
import torch.nn as nn, torch.nn.functional as F
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
assert torch.cuda.is_available(), "No hay GPU: pide una cola nukwa-* y usa el kernel Python-ML-DL"
dev = "cuda"
asignados = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("nodo:", socket.gethostname(), "| GPU:", torch.cuda.get_device_name(0), "| núcleos asignados:", asignados)
print("torch", torch.__version__, "| timm", timm.__version__)

t = pd.read_csv(DATOS / "tabla_modelo.csv", encoding="utf-8-sig")
t["ruta"] = t.apply(lambda r: IMG / str(r["pmcid"]) / str(r["image_file"]), axis=1)
t["existe"] = t["ruta"].map(lambda p: p.exists())
d = t[t.existe & t.clase_modelo.notna()].reset_index(drop=True)
print("imágenes usables:", len(d), "| artículos:", d.pmcid.nunique())
assert len(d) > 1900, "faltan imágenes: ejecuta antes las celdas 1 y 2 del cuaderno 02"
assert (d.groupby("pmcid").fold.nunique() == 1).all(), "un artículo está en dos folds"
clases = sorted(set(d.clase_modelo)); cidx = {c: i for i, c in enumerate(clases)}
yi = np.array([cidx[c] for c in d.clase_modelo]); folds = d.fold.astype(int).values
base = pd.Series(d.clase_modelo).value_counts(normalize=True).max()
print("clases:", clases, "| línea base (mayoritaria):", round(float(base), 3))

LADO = 256
def cargar(ruta):
    im = Image.open(ruta).convert("RGB"); w, h = im.size; l = max(w, h)
    lienzo = Image.new("RGB", (l, l), (255, 255, 255)); lienzo.paste(im, ((l - w) // 2, (l - h) // 2))   # figura completa, rellenada a cuadrado
    return np.asarray(lienzo.resize((LADO, LADO), Image.BICUBIC), dtype=np.uint8)
t0 = time.time()
with ThreadPoolExecutor(max_workers=max(2, asignados)) as ex:
    X = np.stack(list(ex.map(cargar, [str(p) for p in d.ruta])))
Xg = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(dev)       # uint8 en la GPU (~0,4 GB)
del X
print(f"imágenes en la GPU: {tuple(Xg.shape)} ({time.time()-t0:.0f}s)")
print("OK celda 1")


In [ ]:
# Celda 2: aumentos en la GPU, modelo y bucle de entrenamiento con punto de control por época
TAM = 224
MEDIA = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1) * 255
DESV = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1) * 255

def lote_train(idx):
    xb, out = Xg[idx], []
    for i in range(xb.shape[0]):
        s = random.uniform(0.6, 1.0); c = int(LADO * math.sqrt(s))
        x0, y0 = random.randint(0, LADO - c), random.randint(0, LADO - c)
        im = F.interpolate(xb[i:i + 1, :, y0:y0 + c, x0:x0 + c].float(), size=(TAM, TAM), mode="bilinear", align_corners=False)
        out.append(im.flip(3) if random.random() < 0.5 else im)
    return (torch.cat(out) - MEDIA) / DESV

def lote_eval(idx):
    return (F.interpolate(Xg[idx].float(), size=(TAM, TAM), mode="bilinear", align_corners=False) - MEDIA) / DESV

MODELOS = {"resnet50": "resnet50.a1_in1k", "dinov2_vits14": "vit_small_patch14_dinov2.lvd142m"}
os.environ.setdefault("HF_HOME", str(BASE / "hf_cache"))

def crear(corto):
    kw = {"img_size": TAM} if "vit" in MODELOS[corto] else {}
    return timm.create_model(MODELOS[corto], pretrained=True, num_classes=len(clases), **kw).to(dev)

def entrenar_fold(corto, k, epocas, lr, bs=32, semilla=0):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    random.seed(semilla + k); np.random.seed(semilla + k); torch.manual_seed(semilla + k)
    m = crear(corto)
    ids_cab = {id(p) for p in m.get_classifier().parameters()}
    bb = [p for p in m.parameters() if id(p) not in ids_cab]; cab = [p for p in m.parameters() if id(p) in ids_cab]
    opt = torch.optim.AdamW([{"params": bb, "lr": lr}, {"params": cab, "lr": lr * 10}], weight_decay=0.05)
    pasos_ep = len(tr) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr, lr * 10], total_steps=epocas * pasos_ep, pct_start=0.15)
    frec = np.bincount(yi[tr], minlength=len(clases))
    w = torch.tensor((frec.sum() / (len(clases) * frec)) ** 0.5, dtype=torch.float32, device=dev)   # pesos por clase suavizados
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()
    ytr = torch.tensor(yi, device=dev)
    ckpt = SALIDA / f"ckpt_{corto}_f{k}.pt"
    ep0 = 0
    if ckpt.exists():                                        # reanudar si se cortó
        s = torch.load(ckpt, map_location=dev)
        m.load_state_dict(s["m"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"]); ep0 = s["ep"] + 1
        print(f"  reanudando {corto} fold {k} desde la época {ep0}")
    for ep in range(ep0, epocas):
        m.train(); perm = np.random.permutation(tr); perd = ok = n = 0
        for i in range(pasos_ep):
            idx = torch.tensor(perm[i * bs:(i + 1) * bs], device=dev)
            with torch.autocast("cuda", dtype=torch.float16):
                lo = m(lote_train(idx)); loss = crit(lo, ytr[idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perd += float(loss) * len(idx); ok += int((lo.argmax(1) == ytr[idx]).sum()); n += len(idx)
        print(f"  {corto} fold {k} época {ep+1}/{epocas}: pérdida {perd/n:.3f} | acierto de entrenamiento {ok/n:.3f} | {time.time()-T_INICIO:.0f}s", flush=True)
        torch.save({"m": m.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(), "ep": ep}, ckpt)   # punto de control
    m.eval(); P = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(te), 128):
            P.append(torch.softmax(m(lote_eval(torch.tensor(te[i:i + 128], device=dev))).float(), 1).cpu().numpy())
    P = np.concatenate(P)
    df = pd.DataFrame({"pmcid": d.pmcid.values[te], "image_file": d.image_file.values[te], "real": d.clase_modelo.values[te],
                       "prediccion": [clases[j] for j in P.argmax(1)], "fold": k})
    for j, c in enumerate(clases):
        df["p_" + c] = P[:, j].round(5)
    df.to_csv(SALIDA / f"ft_{corto}_fold{k}.csv", index=False, encoding="utf-8")
    ckpt.unlink(missing_ok=True)                              # libera disco (cuota de 10 GB)
    del m, opt; torch.cuda.empty_cache()
    return df
print("OK celda 2")


In [ ]:
# Celda 3: correr los modelos y las particiones (reanudable; se detiene antes del límite de tiempo)
MODELOS_A_CORRER = ["resnet50", "dinov2_vits14"]
FOLDS_A_CORRER = [0, 1, 2, 3, 4]       # para una prueba rápida: [0]
EPOCAS = 10
LR = {"resnet50": 1e-4, "dinov2_vits14": 3e-5}
TIEMPO_MAX_MIN = 50                      # en nukwa-debug (1 h): 50; en nukwa-l40s (6 h): 330

parar = False
for corto in MODELOS_A_CORRER:
    for k in FOLDS_A_CORRER:
        if (SALIDA / f"ft_{corto}_fold{k}.csv").exists():
            print(f"{corto} fold {k}: ya hecho"); continue
        if (time.time() - T_INICIO) / 60 > TIEMPO_MAX_MIN:
            print(f"\nLímite de tiempo ({TIEMPO_MAX_MIN} min). Vuelve a ejecutar esta celda en otra sesión: retoma donde quedó."); parar = True; break
        t1 = time.time()
        df = entrenar_fold(corto, k, EPOCAS, LR[corto])
        acc = (df.real == df.prediccion).mean()
        print(f"== {corto} fold {k} listo en {time.time()-t1:.0f}s | exactitud en la partición de prueba: {acc:.3f}\n", flush=True)
    if parar:
        break
print("OK celda 3")


In [ ]:
# Celda 4: métricas globales (todas las particiones hechas) y comparación con las referencias
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
resumen = {"evaluacion": "5 particiones por articulo (fold); ultima epoca, sin seleccion con datos de prueba", "clases": clases,
           "epocas": EPOCAS, "linea_base_clase_mayoritaria": round(float(base), 4), "modelos": {}}
probs = {}
for corto in MODELOS:
    fs = [SALIDA / f"ft_{corto}_fold{k}.csv" for k in range(5)]
    if not all(f.exists() for f in fs):
        print(corto, "-> faltan particiones:", [f.name for f in fs if not f.exists()]); continue
    o = pd.concat([pd.read_csv(f) for f in fs]).sort_values(["pmcid", "image_file"]).reset_index(drop=True)
    acc, bal, f1 = accuracy_score(o.real, o.prediccion), balanced_accuracy_score(o.real, o.prediccion), f1_score(o.real, o.prediccion, average="macro")
    cm = confusion_matrix(o.real, o.prediccion, labels=clases)
    resumen["modelos"][corto] = {"accuracy": round(float(acc), 4), "accuracy_balanceada": round(float(bal), 4), "f1_macro": round(float(f1), 4), "matriz_confusion": cm.tolist(), "n_imagenes": int(len(o))}
    probs[corto] = o
    o.to_csv(SALIDA / f"predicciones_ft_{corto}.csv", index=False, encoding="utf-8")
    print(f"{corto}: exactitud {acc:.3f} | exactitud balanceada {bal:.3f} | F1 macro {f1:.3f}   (referencia imagen sola con vectores congelados: F1 0,466; clase mayoritaria: exactitud {base:.3f})")
    print(pd.DataFrame(cm, index=clases, columns=clases).to_string(), "\n")
if len(probs) == 2:                       # promedio de probabilidades de los dos modelos
    a, b = probs["resnet50"], probs["dinov2_vits14"]
    assert (a.pmcid.values == b.pmcid.values).all() and (a.image_file.values == b.image_file.values).all()
    pm = (a[["p_" + c for c in clases]].values + b[["p_" + c for c in clases]].values) / 2
    pred = [clases[j] for j in pm.argmax(1)]
    resumen["modelos"]["promedio_resnet_dinov2"] = {"accuracy": round(float(accuracy_score(a.real, pred)), 4), "accuracy_balanceada": round(float(balanced_accuracy_score(a.real, pred)), 4), "f1_macro": round(float(f1_score(a.real, pred, average="macro")), 4)}
    print("promedio de los dos modelos:", resumen["modelos"]["promedio_resnet_dinov2"])
(SALIDA / "metricas_ft.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nArchivos para descargar en", SALIDA, ":")
for p in sorted(SALIDA.glob("*ft*")): print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")
print("OK celda 4. Descarga metricas_ft.json y predicciones_ft_*.csv (y los ft_*_fold*.csv) y pégaselos a Claude.")
